# LLM-based Relevance Validation

This notebook evaluates the relevance of original and synthetic queries
to their corresponding documents using an independent OpenAI language model.

The relevance judge is not informed whether a query is original or synthetic.
It only receives the query and its corresponding document.

Relevance is assessed on a four-point scale from 0 (irrelevant) to
3 (perfectly relevant).

## Imports

In [ ]:
import json
import pandas as pd
from google.colab import userdata
from openai import OpenAI

## API configuration

In [ ]:
api_key = userdata.get("OPENAI_API_KEY").strip()

client = OpenAI(api_key=api_key)

MODEL = "gpt-5.6-luna"

## Load datasets

In [ ]:
news = pd.read_csv("news_full_augmented.csv")
tenders = pd.read_csv("tenders_full_augmented.csv")
abstracts = pd.read_csv("abstracts_full_augmented_clean.csv")

In [ ]:
news.head()

,query_id,corpus_id,score,original_query,document_text,synthetic_query,assigned_task_type
0,7834,7834,1,Verzetsman na 66 jaar begraven,Vandaag wordt in het Noord-Hollandse Heiloo ve...,Hoe werd een Nederlandse oorlogsslachtoffer na...,Task_1
1,124788,124788,1,"Als er kritiek is op de koning, gaat het altij...",Koning Willem-Alexander zit ruim duizend dagen...,Waarom is er kritiek op de financiële uitgaven...,Task_2
2,202109,202109,1,Steeds meer elektrische scooters op de weg,Steeds meer mensen kiezen voor een elektrische...,Klopt de bewering dat elektrische scooters ste...,Task_3
3,93877,93877,1,Militair met drugs gepakt op Curaçao,Op Curaçao is een Nederlandse militair van 25...,Wat is er bekend over de arrestatie van een mi...,Task_1
4,20941,20941,1,Deense grenscontroles vanaf dinsdag,Het Deense parlement heeft ingestemd met het o...,Uitleg over herinvoering grenscontroles en de ...,Task_2


In [ ]:
print(len(news))
print(len(tenders))
print(len(abstracts))

1000
1000
1000


## Define relevance scale

In [ ]:
RELEVANCE_SCALE = """
0 = Irrelevant
The document does not contain information that addresses the information
need expressed by the query.

1 = Related
The document is topically related to the query, but does not provide
sufficient information to satisfy the information need.

2 = Highly relevant
The document provides substantial information that addresses the
information need expressed by the query.

3 = Perfectly relevant
The document directly and comprehensively satisfies the information
need expressed by the query.
"""

## Define final judge prompt

In [ ]:
SYSTEM_PROMPT = f"""
You are an information retrieval relevance assessor.

Your task is to judge how relevant a document is to a search query.

Evaluate ONLY whether the document contains information that addresses
the information need expressed by the query.

Do NOT judge:
- whether the query is well-written;
- whether the query is grammatically correct;
- whether the query sounds natural;
- whether the query was written by a human or generated by an AI system.

Do not assume that the query is relevant simply because it is paired
with the document.

Use the following relevance scale:

{RELEVANCE_SCALE}

Provide a brief explanation for your judgment.

The output must follow the specified JSON schema.
"""

## Define structured output scheme

In [ ]:
RELEVANCE_SCHEMA = {
    "type": "object",
    "properties": {
        "relevance_score": {
            "type": "integer",
            "enum": [0, 1, 2, 3]
        },
        "reason": {
            "type": "string"
        }
    },
    "required": ["relevance_score", "reason"],
    "additionalProperties": False
}

## Define judge_relevance()

In [ ]:


def judge_relevance(query, document):
    response = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        input=f"""
DOCUMENT:
{document}

QUERY:
{query}
""",
        text={
            "format": {
                "type": "json_schema",
                "name": "relevance_judgment",
                "description": "Relevance judgment for a query-document pair.",
                "schema": RELEVANCE_SCHEMA,
                "strict": True
            }
        }
    )

    result = json.loads(response.output_text)

    return {
        "relevance_score": result["relevance_score"],
        "reason": result["reason"]
    }

## Prepare final evaluation data

In [ ]:
news["dataset"] = "News"
tenders["dataset"] = "Tenders"
abstracts["dataset"] = "Abstracts"

relevance_df = pd.concat(
    [news, tenders, abstracts],
    ignore_index=True
)

## Check dataset sizes

In [ ]:
relevance_df.head()

,query_id,corpus_id,score,original_query,document_text,synthetic_query,assigned_task_type,dataset
0,7834,7834,1,Verzetsman na 66 jaar begraven,Vandaag wordt in het Noord-Hollandse Heiloo ve...,Hoe werd een Nederlandse oorlogsslachtoffer na...,Task_1,News
1,124788,124788,1,"Als er kritiek is op de koning, gaat het altij...",Koning Willem-Alexander zit ruim duizend dagen...,Waarom is er kritiek op de financiële uitgaven...,Task_2,News
2,202109,202109,1,Steeds meer elektrische scooters op de weg,Steeds meer mensen kiezen voor een elektrische...,Klopt de bewering dat elektrische scooters ste...,Task_3,News
3,93877,93877,1,Militair met drugs gepakt op Curaçao,Op Curaçao is een Nederlandse militair van 25...,Wat is er bekend over de arrestatie van een mi...,Task_1,News
4,20941,20941,1,Deense grenscontroles vanaf dinsdag,Het Deense parlement heeft ingestemd met het o...,Uitleg over herinvoering grenscontroles en de ...,Task_2,News


In [ ]:
len(relevance_df)

3000

In [ ]:
relevance_df.columns

Index(['query_id', 'corpus_id', 'score', 'original_query', 'document_text',
       'synthetic_query', 'assigned_task_type', 'dataset'],
      dtype='object')

## Run relevance labelling with checkpointing

In [ ]:
import os
import pandas as pd
import time

OUTPUT_PATH = "relevance_labels_final.csv"

results = []

In [ ]:
def label_row(row, query_type):

    if query_type == "original":
        query = row["original_query"]
    elif query_type == "synthetic":
        query = row["synthetic_query"]
    else:
        raise ValueError(f"Unknown query type: {query_type}")

    judgment = judge_relevance(
        query=query,
        document=row["document_text"]
    )

    return {
        "query_id": row["query_id"],
        "corpus_id": row["corpus_id"],
        "dataset": row["dataset"],
        "query_type": query_type,
        "assigned_task_type": row["assigned_task_type"],
        "query": query,
        "relevance_score": judgment["relevance_score"],
        "reason": judgment["reason"]
    }

In [ ]:
cases = []

for _, row in relevance_df.iterrows():

    cases.append((row, "original"))
    cases.append((row, "synthetic"))

print("Total cases:", len(cases))

Total cases: 6000


In [ ]:
CHECKPOINT_PATH = "relevance_labels_checkpoint.csv"

if os.path.exists(CHECKPOINT_PATH):
    results_df = pd.read_csv(CHECKPOINT_PATH)

    completed = set(
        zip(
            results_df["query_id"],
            results_df["query_type"]
        )
    )

    print(f"Existing labels: {len(results_df)}")
else:
    results_df = pd.DataFrame()
    completed = set()

    print("Starting from zero.")

Starting from zero.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
CHECKPOINT_PATH = (
    "/content/drive/MyDrive/"
    "relevance_labels_checkpoint.csv"
)

In [ ]:
for i, (row, query_type) in enumerate(cases, start=1):

    key = (row["query_id"], query_type)

    # Skip if already completed
    if key in completed:
        continue

    try:
        result = label_row(row, query_type)

        results_df = pd.concat(
            [
                results_df,
                pd.DataFrame([result])
            ],
            ignore_index=True
        )

        # Save after EVERY label
        results_df.to_csv(
            CHECKPOINT_PATH,
            index=False
        )

        print(
            f"[{i}/{len(cases)}] "
            f"{row['dataset']} | "
            f"{query_type} | "
            f"{row['query_id']} | "
            f"Score: {result['relevance_score']}"
        )

    except Exception as e:
        print(
            f"ERROR at {i}/{len(cases)} "
            f"| {row['dataset']} "
            f"| {row['query_id']} "
            f"| {query_type}"
        )
        print(e)

        # Stop so the error can be inspected
        raise

Streaminguitvoer ingekort tot de laatste 5000 regels.
[1001/6000] News | original | 194567 | Score: 3
[1002/6000] News | synthetic | 194567 | Score: 3
[1003/6000] News | original | 25427 | Score: 3
[1004/6000] News | synthetic | 25427 | Score: 3
[1005/6000] News | original | 93453 | Score: 3
[1006/6000] News | synthetic | 93453 | Score: 3
[1007/6000] News | original | 119274 | Score: 3
[1008/6000] News | synthetic | 119274 | Score: 2
[1009/6000] News | original | 37162 | Score: 3
[1010/6000] News | synthetic | 37162 | Score: 3
[1011/6000] News | original | 182302 | Score: 3
[1012/6000] News | synthetic | 182302 | Score: 3
[1013/6000] News | original | 156831 | Score: 3
[1014/6000] News | synthetic | 156831 | Score: 2
[1015/6000] News | original | 151511 | Score: 3
[1016/6000] News | synthetic | 151511 | Score: 3
[1017/6000] News | original | 107218 | Score: 3
[1018/6000] News | synthetic | 107218 | Score: 3
[1019/6000] News | original | 153856 | Score: 3
[1020/6000] News | synthetic | 

## Export final labels

In [ ]:
 relevance_df = pd.read_csv("/content/drive/MyDrive/relevance_labels_checkpoint.csv")

print(relevance_df.shape)
print(relevance_df["query_type"].value_counts())
print(relevance_df["dataset"].value_counts())
print(relevance_df["relevance_score"].value_counts().sort_index())

(6000, 8)
query_type
original     3000
synthetic    3000
Name: count, dtype: int64
dataset
News         2000
Tenders      2000
Abstracts    2000
Name: count, dtype: int64
relevance_score
0      48
1     735
2     752
3    4465
Name: count, dtype: int64


In [ ]:
relevance_summary = (
    relevance_df
    .groupby(["dataset", "query_type"])["relevance_score"]
    .agg(["count", "mean", "median", "std"])
    .round(3)
)

display(relevance_summary)

count   mean  median    std
dataset   query_type                             
Abstracts original     1000  2.449     3.0  0.836
          synthetic    1000  2.344     3.0  0.816
News      original     1000  2.984     3.0  0.133
          synthetic    1000  2.905     3.0  0.355
Tenders   original     1000  2.466     3.0  0.844
          synthetic    1000  2.486     3.0  0.784

In [ ]:
relevance_distribution = (
    relevance_df
    .groupby(["dataset", "query_type", "relevance_score"])
    .size()
    .unstack(fill_value=0)
)

display(relevance_distribution)

relevance_score        0    1    2    3
dataset   query_type                   
Abstracts original    18  171  155  656
          synthetic    1  217  219  563
News      original     0    1   14  985
          synthetic    2   14   61  923
Tenders   original    26  153  150  671
          synthetic    1  179  153  667

In [ ]:
relevance_distribution_pct = (
    relevance_df
    .groupby(["dataset", "query_type"])["relevance_score"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .mul(100)
    .round(1)
)

display(relevance_distribution_pct)

relevance_score         0     1     2     3
dataset   query_type                       
Abstracts original    1.8  17.1  15.5  65.6
          synthetic   0.1  21.7  21.9  56.3
News      original    0.0   0.1   1.4  98.5
          synthetic   0.2   1.4   6.1  92.3
Tenders   original    2.6  15.3  15.0  67.1
          synthetic   0.1  17.9  15.3  66.7

In [ ]:
paired = relevance_df.pivot(
    index=["dataset", "query_id"],
    columns="query_type",
    values="relevance_score"
).reset_index()

paired["difference"] = (
    paired["synthetic"] - paired["original"]
)

display(
    paired.groupby("dataset")["difference"]
    .value_counts()
    .unstack(fill_value=0)
)

difference,-3,-2,-1,0,1,2,3
dataset,,,,,,,
Abstracts,1,101,175,531,117,67,8
News,2,14,61,908,14,1,0
Tenders,0,101,121,540,141,89,8


In [ ]:
paired_summary = (
    paired
    .groupby("dataset")["difference"]
    .agg(
        mean="mean",
        median="median",
        std="std",
        lower=lambda x: (x < 0).sum(),
        equal=lambda x: (x == 0).sum(),
        higher=lambda x: (x > 0).sum()
    )
    .round(3)
)

display(paired_summary)

,mean,median,std,lower,equal,higher
dataset,,,,,,
Abstracts,-0.105,0.0,1.017,277,531,192
News,-0.079,0.0,0.383,77,908,15
Tenders,0.020,0.0,1.046,222,540,238


In [ ]:
from scipy.stats import wilcoxon

for dataset in paired["dataset"].unique():
    subset = paired[paired["dataset"] == dataset]

    stat, p = wilcoxon(
        subset["original"],
        subset["synthetic"]
    )

    print(
        dataset,
        f"W = {stat:.1f}, p = {p:.4g}"
    )

Abstracts W = 46086.0, p = 0.001432
News W = 615.0, p = 1.794e-10
Tenders W = 52019.0, p = 0.7182


In [ ]:
stat, p = wilcoxon(
    paired["original"],
    paired["synthetic"]
)

print(f"Overall: W = {stat:.1f}, p = {p:.4g}")

Overall: W = 229984.0, p = 0.000677
